In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

In [2]:
client = bigquery.Client(project="pacey32-agency")

In [3]:
sql = """
WITH profile AS (
    SELECT
        playerId,
        ANY_VALUE(player_name) AS player,
        ANY_VALUE(age) AS age,
        ANY_VALUE(position) AS position,
        ANY_VALUE(heightInCentimeters) AS height_cm,
        ANY_VALUE(weightInKilograms) AS weight_kg,
        ANY_VALUE(draftRound) AS draftRound,
        ANY_VALUE(draftOverall) AS draftPick,
        ANY_VALUE(draftYear) AS draftYear,
        ANY_VALUE(draftTeam) AS draftTeam,
        ANY_VALUE(shoots_catches) AS shoots_catches,
        ANY_VALUE(birth_country) AS birthCountry
    FROM `pacey32-agency.Comparison.01_PlayerProfile`
    WHERE activeFlag = 1
    GROUP BY playerId
),

stats AS (
    SELECT
        playerId,
        season,
        SUM(games_played) AS games,
        SUM(goals) AS goals,
        SUM(assists) AS assists,
        SUM(points) AS points,
        SAFE_DIVIDE(SUM(goals), SUM(games_played)) AS goals_per_game,
        SAFE_DIVIDE(SUM(assists), SUM(games_played)) AS assists_per_game,
        SAFE_DIVIDE(SUM(points), SUM(games_played)) AS points_per_game,
        SAFE_DIVIDE(SUM(toi_minutes), SUM(games_played)) AS avg_toi_minutes,
        SAFE_DIVIDE(SUM(goals_per_60 * toi_minutes), SUM(toi_minutes)) AS goals_per_60,
        SAFE_DIVIDE(SUM(assists_per_60 * toi_minutes), SUM(toi_minutes)) AS assists_per_60,
        SAFE_DIVIDE(SUM(points_per_60 * toi_minutes), SUM(toi_minutes)) AS points_per_60
    FROM `pacey32-agency.Comparison.03_PlayerSeasonStats`
    WHERE seasonPart = 'RegularSeason'
    GROUP BY playerId, season
),

latest AS (
    SELECT *
    FROM stats
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY playerId
        ORDER BY season DESC
    ) = 1
)

SELECT
    p.*,
    s.season,
    s.games,
    s.goals,
    s.assists,
    s.points,
    s.goals_per_game,
    s.assists_per_game,
    s.points_per_game,
    s.avg_toi_minutes,
    s.goals_per_60,
    s.assists_per_60,
    s.points_per_60
FROM profile p
LEFT JOIN latest s
    ON p.playerId = CAST(s.playerId AS INT64)
"""

df_players = client.query(sql).to_dataframe()

#display(df_players.head())
#print(f"Players: {len(df_players):,}")

In [4]:
sql_location = """
SELECT *
FROM `pacey32-agency.EventLocations.10_PlayerPerformanceLocationLatest`
"""

df_location = client.query(sql_location).to_dataframe()

print(f"Players with location data: {len(df_location):,}")
display(df_location.head())

Players with location data: 1,608


,playerId,player,latest_season,seasons_used,latest_games_played,latest_toi_minutes,latest_shot_pct_close_left,latest_shot_pct_close_centre,latest_shot_pct_close_right,latest_shot_pct_medium_left,...,weighted3_giveaways_per_game,weighted3_giveaways_per_60,latest_takeaways_per_game,latest_takeaways_per_60,weighted3_takeaways_per_game,weighted3_takeaways_per_60,latest_hits_per_game,latest_hits_per_60,weighted3_hits_per_game,weighted3_hits_per_60
0,8475765,Vladimir Tarasenko,20252026,3,75,1120.4,0.046099,0.304965,0.039007,0.159574,...,0.794693,3.191109,0.160000,0.642628,0.175684,0.702508,0.800000,3.213138,0.833289,3.329992
1,8480289,Morgan Barron,20252026,3,65,832.0,0.017751,0.366864,0.041420,0.230769,...,0.451668,2.221812,0.261538,1.225962,0.277739,1.403400,2.030769,9.519231,1.911876,9.488731
2,8482092,Ridly Greig,20252026,3,77,1285.4,0.032258,0.391129,0.036290,0.145161,...,0.561587,2.023721,0.259740,0.933562,0.271989,0.991913,1.259740,4.527773,1.418784,5.156315
3,8478449,Roope Hintz,20252026,3,53,919.1,0.009709,0.495146,0.024272,0.140777,...,0.771970,2.688893,0.094340,0.326406,0.140775,0.491808,1.113208,3.851594,0.963719,3.348171
4,8480336,Sean Walker,20252026,3,81,1766.8,0.004454,0.155902,0.008909,0.062361,...,1.023908,2.960821,0.407407,1.120670,0.433363,1.290109,1.753086,4.822278,1.594791,4.621156


In [5]:
df_players["playerId"] = pd.to_numeric(df_players["playerId"], errors="coerce").astype("Int64")
df_location["playerId"] = pd.to_numeric(df_location["playerId"], errors="coerce").astype("Int64")

df_model = df_players.merge(
    df_location.drop(columns=["player"], errors="ignore"),
    on="playerId",
    how="left"
)

print(f"Players: {len(df_model):,}")
print(f"Players with location data: {df_model['latest_season'].notna().sum():,}")

display(df_model.head())

Players: 1,076
Players with location data: 950


,playerId,player,age,position,height_cm,weight_kg,draftRound,draftPick,draftYear,draftTeam,...,weighted3_giveaways_per_game,weighted3_giveaways_per_60,latest_takeaways_per_game,latest_takeaways_per_60,weighted3_takeaways_per_game,weighted3_takeaways_per_60,latest_hits_per_game,latest_hits_per_60,weighted3_hits_per_game,weighted3_hits_per_60
0,8475692,Mats Zuccarello,39,R,173,82,<NA>,<NA>,<NA>,None,...,1.013854,3.223438,0.220339,0.709091,0.293368,0.927844,0.271186,0.872727,0.320265,1.013701
1,8477451,Ryan Hartman,32,R,183,89,1,30,2013,CHI,...,0.849199,3.126556,0.302632,1.084394,0.325414,1.194484,0.855263,3.064592,0.914528,3.373035
2,8477992,Jonas Johansson,31,G,196,100,3,61,2014,BUF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8479407,Jesper Bratt,28,L,178,79,6,162,2016,NJD,...,0.993330,3.166668,0.292683,0.935916,0.279600,0.888948,0.914634,2.924737,0.960238,3.055847
4,8480014,Gabriel Vilardi,27,C,191,98,1,11,2017,LAK,...,0.680957,2.201521,0.170732,0.546626,0.226165,0.745273,0.256098,0.819939,0.268210,0.874734


In [9]:
feature_groups = {
    "profile": [
        "age", "height_cm", "weight_kg", "draftPick"
    ],

    "production": [
        "goals_per_game", "assists_per_game", "points_per_game",
        "avg_toi_minutes", "goals_per_60", "assists_per_60", "points_per_60"
    ],

    "shot_location_latest": [
        f"latest_shot_pct_{z}_{s}"
        for z in ["close", "medium", "far"]
        for s in ["left", "centre", "right"]
    ],

    "shot_location_weighted3": [
        f"weighted3_shot_pct_{z}_{s}"
        for z in ["close", "medium", "far"]
        for s in ["left", "centre", "right"]
    ],

    "shooting_efficiency_latest": [
        f"latest_shooting_pct_{z}_{s}"
        for z in ["close", "medium", "far"]
        for s in ["left", "centre", "right"]
    ],

    "shooting_efficiency_weighted3": [
        f"weighted3_shooting_pct_{z}_{s}"
        for z in ["close", "medium", "far"]
        for s in ["left", "centre", "right"]
    ],

    "shot_volume": [
        "latest_shots_per_game", "latest_shots_per_60",
        "weighted3_shots_per_game", "weighted3_shots_per_60"
    ],

    "faceoffs": [
        "latest_faceoff_pct", "latest_faceoffs_per_game", "latest_faceoffs_per_60",
        "latest_faceoff_wins_per_game", "latest_faceoff_wins_per_60",
        "weighted3_faceoff_pct", "weighted3_faceoffs_per_game", "weighted3_faceoffs_per_60",
        "weighted3_faceoff_wins_per_game", "weighted3_faceoff_wins_per_60"
    ],

    "discipline": [
        "latest_penalties_per_game", "latest_penalties_per_60",
        "weighted3_penalties_per_game", "weighted3_penalties_per_60"
    ],

    "puck_management": [
        "latest_giveaways_per_game", "latest_giveaways_per_60",
        "latest_takeaways_per_game", "latest_takeaways_per_60",
        "weighted3_giveaways_per_game", "weighted3_giveaways_per_60",
        "weighted3_takeaways_per_game", "weighted3_takeaways_per_60"
    ],

    "physical": [
        "latest_hits_per_game", "latest_hits_per_60",
        "weighted3_hits_per_game", "weighted3_hits_per_60"
    ]
}

In [10]:
for group, cols in feature_groups.items():
    missing = [c for c in cols if c not in df_model.columns]
    print(f"{group:20} {len(cols):2} features | missing: {missing}")

profile               4 features | missing: []
production            7 features | missing: []
shot_location_latest  9 features | missing: []
shot_location_weighted3  9 features | missing: []
shooting_efficiency_latest  9 features | missing: []
shooting_efficiency_weighted3  9 features | missing: []
shot_volume           4 features | missing: []
faceoffs             10 features | missing: []
discipline            4 features | missing: []
puck_management       8 features | missing: []
physical              4 features | missing: []


In [11]:
all_features = [c for cols in feature_groups.values() for c in cols]

print(f"Total features: {len(all_features)}")
print(f"Unique features: {len(set(all_features))}")

Total features: 77
Unique features: 77


In [12]:
import numpy as np
from sklearn.preprocessing import StandardScaler

df_compare = df_model.dropna(subset=["playerId", "position"]).copy()

X = df_compare[all_features].copy()

# Fill missing values with each feature's median
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Players: {len(df_compare):,}")
print(f"Features: {X_scaled.shape[1]}")
print(f"Matrix: {X_scaled.shape}")

Players: 1,076
Features: 77
Matrix: (1076, 77)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def find_comparables(player_id, n=10):
    idx = df_compare.index[df_compare["playerId"] == player_id]
    if len(idx) == 0:
        print("Player not found")
        return None

    row = df_compare.index.get_loc(idx[0])
    target = df_compare.iloc[row]
    same_position = df_compare["position"].eq(target["position"]).values

    sims = cosine_similarity(X_scaled[row].reshape(1, -1), X_scaled)[0]

    result = df_compare.loc[same_position, [
        "playerId", "player", "age", "position", "shoots_catches", "draftPick"
    ]].copy()

    result["similarity"] = sims[same_position] * 100

    # Remove selected player
    result = result[result["playerId"] != player_id]

    return result.sort_values("similarity", ascending=False).head(n).reset_index(drop=True)

In [14]:
display(find_comparables(8480069, 10))

,playerId,player,age,position,shoots_catches,draftPick,similarity
0,8474578,Erik Karlsson,36,D,R,15,88.412303
1,8480803,Evan Bouchard,27,D,R,10,87.619994
2,8478460,Zach Werenski,29,D,L,8,82.802638
3,8474600,Roman Josi,36,D,L,38,80.903114
4,8480800,Quinn Hughes,27,D,L,7,80.882584
5,8479323,Adam Fox,28,D,R,66,80.864488
6,8480036,Miro Heiskanen,27,D,L,3,79.773733
7,8485366,Matthew Schaefer,19,D,L,1,77.018383
8,8479345,Jakob Chychrun,28,D,L,16,76.754910
9,8482122,Brock Faber,24,D,R,45,74.906301


In [15]:
display(find_comparables(8484153, 10))

,playerId,player,age,position,shoots_catches,draftPick,similarity
0,8478403,Jack Eichel,30,C,R,2,88.200220
1,8479318,Auston Matthews,29,C,L,1,85.100520
2,8484801,Macklin Celebrini,20,C,L,1,83.713227
3,8478427,Sebastian Aho,29,C,L,35,82.804889
4,8482665,Matty Beniers,24,C,L,2,82.103970
5,8477493,Aleksander Barkov,31,C,L,2,82.097530
6,8477951,Nick Schmaltz,30,C,R,20,81.130446
7,8483493,Frank Nazar,22,C,R,13,80.822755
8,8477500,Bo Horvat,31,C,L,9,80.818165
9,8480018,Nick Suzuki,27,C,R,13,80.526104


In [16]:
def explain_comparison(player1_id, player2_id):
    i1 = df_compare.index.get_loc(df_compare.index[df_compare["playerId"] == player1_id][0])
    i2 = df_compare.index.get_loc(df_compare.index[df_compare["playerId"] == player2_id][0])

    z1 = X_scaled[i1]
    z2 = X_scaled[i2]

    result = pd.DataFrame({
        "feature": all_features,
        "player1_value": df_compare.iloc[i1][all_features].values,
        "player2_value": df_compare.iloc[i2][all_features].values,
        "standardised_difference": np.abs(z1 - z2)
    })

    return result.sort_values("standardised_difference").reset_index(drop=True)

In [17]:
comparison = explain_comparison(8484153, 8478403)
display(comparison.head(20))

,feature,player1_value,player2_value,standardised_difference
0,weight_kg,94,94,0.000000
1,draftPick,2,2,0.000000
2,latest_shooting_pct_far_centre,0.0,0.0,0.000000
3,latest_shot_pct_far_left,0.086253,0.085774,0.003765
4,weighted3_shooting_pct_close_centre,0.127285,0.128369,0.014101
5,weighted3_shooting_pct_far_left,0.021875,0.021094,0.016630
6,weighted3_shooting_pct_medium_centre,0.071907,0.068647,0.047170
7,latest_shooting_pct_close_centre,0.121212,0.125,0.047685
8,latest_faceoffs_per_60,44.043267,45.127159,0.067023
9,weighted3_shot_pct_far_left,0.077229,0.087627,0.083456


In [18]:
display(comparison.tail(20).sort_values("standardised_difference", ascending=False))

,feature,player1_value,player2_value,standardised_difference
76,age,22,30,1.828043
75,assists_per_game,0.557143,0.851351,1.474776
74,latest_giveaways_per_game,1.114286,1.621622,1.409947
73,weighted3_giveaways_per_game,0.976029,1.435568,1.375324
72,weighted3_shots_per_game,4.655263,6.270828,1.230467
71,latest_shot_pct_close_centre,0.355795,0.200837,1.167888
70,weighted3_shot_pct_medium_centre,0.145847,0.082349,1.158727
69,weighted3_shot_pct_close_centre,0.35492,0.212989,1.093544
68,latest_shot_pct_medium_centre,0.137466,0.079498,1.091472
67,latest_shot_pct_medium_left,0.180593,0.25523,0.971451
